# Filtering and Smoothing in Discrete Time
### Seminar: Digital Twins — Implementation Notebook

*Freie Universität Berlin · July 2026*

---

This notebook is the accompanying implementation for the seminar talk on **Filtering and Smoothing in Discrete Time**.
All three algorithms are implemented in Python and applied to the **Hidden Market Regime** running example from the slides.

**Contents**

1. [Hidden Markov Model](#1.-Hidden-Markov-Model)
2. [Bayesian Filtering](#2.-Bayesian-Filtering)
3. [Bayesian Smoothing](#3.-Bayesian-Smoothing)
4. [Viterbi Algorithm](#4.-Viterbi-Algorithm)
5. [Comparison](#5.-Comparison)

In [ ]:
import numpy as np
from IPython.display import Image, display

np.set_printoptions(precision=4, suppress=True)

## 1. Hidden Markov Model

A **Hidden Markov Model (HMM)** on measurable spaces $(S_X, \mathcal{S}_X)$ and $(S_Y, \mathcal{S}_Y)$ consists of:

| Component | Symbol | Description |
|-----------|--------|-------------|
| Initial distribution | $X_0 \sim \mu_0$ | Prior over the first hidden state |
| Transition kernel | $X_k \mid (X_{k-1}=x_{k-1}) \sim G(\mathrm{d}x_k \mid x_{k-1})$ | Latent-state dynamics |
| Emission kernel | $Y_k \mid (X_k=x_k) \sim H(\mathrm{d}y_k \mid x_k)$ | How observations are generated |

Under a density assumption, $G(\mathrm{d}x_k \mid x_{k-1}) = p(x_k \mid x_{k-1})\,\lambda_X(\mathrm{d}x_k)$ and analogously for $H$. For a **finite-state** model, $\lambda_X$ and $\lambda_Y$ are counting measures.

The two key structural properties of the joint law are:

1. **Markov states (forward and backward):**
$$\mathbb{P}(X_k \in A \mid X_0,\ldots,X_{k-1}, Y_1,\ldots,Y_{k-1}) = \mathbb{P}(X_k \in A \mid X_{k-1})$$
$$\mathbb{P}(X_{k-1} \in A \mid X_{k:T}, Y_{k:T}) = \mathbb{P}(X_{k-1} \in A \mid X_k)$$

2. **Conditionally independent observations:**
$$\mathbb{P}(Y_k \in B \mid X_{0:k}, Y_{1:k-1}) = \mathbb{P}(Y_k \in B \mid X_k)$$

---

### Example: Hidden Market Regimes

We model daily stock-market dynamics as a finite-state HMM.  
The **latent market regime** is $X_k \in \{\text{Bullish},\; \text{Neutral},\; \text{Bearish}\}$  
and the **observed daily return category** is $Y_k \in \{\text{Large Down},\; \text{Down},\; \text{Flat},\; \text{Up},\; \text{Large Up}\}$.

The **transition matrix** $G$ (with $G_{ij} = p(X_k = j \mid X_{k-1} = i)$, rows sum to 1) and **emission matrix** $H$ (with $H_{io} = p(Y_k = o \mid X_k = i)$, rows sum to 1) are:

$$G = \begin{pmatrix} 0.85 & 0.10 & 0.05 \\ 0.15 & 0.70 & 0.15 \\ 0.05 & 0.10 & 0.85 \end{pmatrix}, \qquad H = \begin{pmatrix} 0.02 & 0.08 & 0.20 & 0.45 & 0.25 \\ 0.10 & 0.20 & 0.40 & 0.20 & 0.10 \\ 0.25 & 0.45 & 0.20 & 0.08 & 0.02 \end{pmatrix}$$

Initial distribution: $\pi_0 = (0.40,\; 0.40,\; 0.20)^\top$.

> **Interpretation.** A Bullish regime produces Up / Large Up returns with total probability 0.70; a Bearish regime produces Down / Large Down with total probability 0.70. The diagonal of $G$ is $\geq 0.70$, reflecting regime persistence.

In [ ]:
# ── State and observation spaces ──────────────────────────────────────────────
states    = ['Bullish', 'Neutral', 'Bearish']
obs_names = ['Large Down', 'Down', 'Flat', 'Up', 'Large Up']
n_states, n_obs = 3, 5

# ── Model parameters ──────────────────────────────────────────────────────────
# Initial distribution μ₀
pi_0 = np.array([0.40, 0.40, 0.20])

# Transition matrix G:  G[i, j] = p(X_k = j | X_{k-1} = i)
G = np.array([[0.85, 0.10, 0.05],
              [0.15, 0.70, 0.15],
              [0.05, 0.10, 0.85]])

# Emission matrix H:  H[i, o] = p(Y_k = o | X_k = i)
H = np.array([[0.02, 0.08, 0.20, 0.45, 0.25],
              [0.10, 0.20, 0.40, 0.20, 0.10],
              [0.25, 0.45, 0.20, 0.08, 0.02]])

# Observed sequence (indices into obs_names)
# T = 4: used for filtering and smoothing examples (matches the figures)
obs_seq_fs   = [3, 4, 3, 2]              # Up, Large Up, Up, Flat
# T = 8: used for the full Viterbi example (matches the trellis figure)
obs_seq_full = [3, 4, 3, 2, 1, 0, 1, 2]  # Up, Large Up, Up, Flat, Down, Large Down, Down, Flat

print('Transition matrix G (rows = from-state, cols = to-state):')
print(G)
print('Row sums:', G.sum(axis=1))
print()
print('Emission matrix H (rows = state, cols = observation):')
print(H)
print('Row sums:', H.sum(axis=1))

In [ ]:
# Emission distribution g(y_k | x_k) per regime
display(Image('emission_distributions.png'))

## 2. Bayesian Filtering

The **filtering distribution** at time $k$ is
$$\pi_k(x_k) := p(x_k \mid y_{1:k}),$$
the conditional law of the hidden state given all observations *up to and including* time $k$.

### Prediction–Update Recursion

Starting from $\pi_0 = p_0$, for $k \geq 1$:

**Prediction step** (Chapman–Kolmogorov):
$$\hat{\pi}_k(x_k) := p(x_k \mid y_{1:k-1}) = \int_{S_X} p(x_k \mid x_{k-1})\,\pi_{k-1}(x_{k-1})\,\mathrm{d}x_{k-1}$$

**Update step** (Bayes):
$$\pi_k(x_k) = p(x_k \mid y_{1:k}) = \frac{p(y_k \mid x_k)\,\hat{\pi}_k(x_k)}{\int_{S_X} p(y_k \mid x_k')\,\hat{\pi}_k(x_k')\,\mathrm{d}x_k'}$$

### Finite-State Implementation

For a finite state space with transition matrix $G$ and emission matrix $H$:

- **Prediction:** $\hat{\pi}_k = G^\top \pi_{k-1}$ (matrix–vector product, $\hat{\pi}_k(j) = \sum_i G_{ij}\,\pi_{k-1}(i)$)
- **Update:** $\pi_k(x) \propto H_{x,\,y_k} \cdot \hat{\pi}_k(x)$ (pointwise multiply, then normalise)

The filter is an *online* algorithm: each new observation $y_k$ requires only the previous filter $\pi_{k-1}$.

In [ ]:
def bayesian_filter(obs_seq, pi_0, G, H):
    """
    Bayesian filtering: prediction-update recursion.

    Parameters
    ----------
    obs_seq : list[int]   observation indices, length T
    pi_0    : ndarray(n)  initial distribution
    G       : ndarray(n,n) transition matrix, G[i,j] = p(X_k=j | X_{k-1}=i)
    H       : ndarray(n,m) emission matrix,   H[i,o] = p(Y_k=o | X_k=i)

    Returns
    -------
    pi_pred : ndarray(T, n)  prediction distributions  hat_pi_k
    pi_filt : ndarray(T, n)  filter distributions       pi_k
    """
    T, n = len(obs_seq), len(pi_0)
    pi_pred = np.zeros((T, n))
    pi_filt = np.zeros((T, n))

    prev = pi_0.copy()
    for k in range(T):
        # Prediction:  hat_pi_k = G^T pi_{k-1}
        pi_pred[k] = G.T @ prev
        # Update:      pi_k  proportional to  H[:, y_k] * hat_pi_k
        unnorm     = H[:, obs_seq[k]] * pi_pred[k]
        pi_filt[k] = unnorm / unnorm.sum()
        prev       = pi_filt[k].copy()

    return pi_pred, pi_filt


pi_pred, pi_filt = bayesian_filter(obs_seq_fs, pi_0, G, H)
T_fs = len(obs_seq_fs)

print('Filtering results (T = 4):')
print(f"{'k':>3}  {'y_k':>10}  {'pi_k(Bull)':>12}  {'pi_k(Neut)':>12}  {'pi_k(Bear)':>12}")
print('-' * 55)
for k in range(T_fs):
    y = obs_names[obs_seq_fs[k]]
    print(f"{k+1:>3}  {y:>10}  {pi_filt[k,0]:>12.4f}  {pi_filt[k,1]:>12.4f}  {pi_filt[k,2]:>12.4f}")

In [ ]:
# Filtering distributions pi_k (solid bars) vs prediction hat_pi_k (hatched bars)
display(Image('filtering_distributions.png'))

## 3. Bayesian Smoothing

The **smoothing distribution** at time $k$ is
$$\pi_k^T(x_k) := p(x_k \mid y_{1:T}),$$
the conditional law of $X_k$ given the *full* observation record $y_{1:T}$. For $k < T$ this is strictly more informative than the filter $\pi_k$.

### Backward Recursion

**Initialise:** $\pi_T^T = \pi_T$

**Update backwards** for $k = T-1, \ldots, 1$:
$$\pi_k^T(x_k) = \pi_k(x_k) \cdot \int_{S_X} \frac{p(x_{k+1} \mid x_k)\; \pi_{k+1}^T(x_{k+1})}{\hat{\pi}_{k+1}(x_{k+1})}\;\mathrm{d}x_{k+1}$$

### Proof sketch

$$p(x_k \mid y_{1:T}) = \int p(x_k, x_{k+1} \mid y_{1:T})\,\mathrm{d}x_{k+1} \quad \text{(marginalise)}$$
$$= \int p(x_k \mid x_{k+1}, y_{1:k})\; p(x_{k+1} \mid y_{1:T})\,\mathrm{d}x_{k+1} \quad \text{(backward Markov + chain rule)}$$
$$= \int \frac{p(x_{k+1} \mid x_k)\; p(x_k \mid y_{1:k})}{p(x_{k+1} \mid y_{1:k})}\; p(x_{k+1} \mid y_{1:T})\,\mathrm{d}x_{k+1} \quad \text{(Bayes' theorem)}$$

### Finite-State Implementation

Let $r_{k+1}(j) = \pi_{k+1}^T(j) \;/\; \hat{\pi}_{k+1}(j)$. Then:
$$\pi_k^T(i) \propto \pi_k(i) \cdot \sum_j G_{ij}\; r_{k+1}(j) = \pi_k(i) \cdot (G\, r_{k+1})(i)$$

This requires the stored filter $\pi_k$ and predictor $\hat{\pi}_{k+1}$ from the forward pass.

In [ ]:
def bayesian_smooth(pi_pred, pi_filt, G):
    """
    Bayesian smoothing: backward recursion (RTS smoother for finite-state HMMs).

    Parameters
    ----------
    pi_pred : ndarray(T, n)  predictions  hat_pi_k  (from bayesian_filter)
    pi_filt : ndarray(T, n)  filters       pi_k      (from bayesian_filter)
    G       : ndarray(n, n)  transition matrix, G[i,j] = p(X_{k+1}=j | X_k=i)

    Returns
    -------
    pi_smooth : ndarray(T, n)  smoother distributions pi_k^T
    """
    T, n = pi_filt.shape
    pi_smooth      = np.zeros((T, n))
    pi_smooth[-1]  = pi_filt[-1].copy()   # initialise: pi_T^T = pi_T

    for k in range(T - 2, -1, -1):
        # ratio r_{k+1}(j) = pi_{k+1}^T(j) / hat_pi_{k+1}(j)
        ratio         = pi_smooth[k+1] / np.maximum(pi_pred[k+1], 1e-12)
        # pi_k^T(i)  proportional to  pi_k(i) * (G @ ratio)(i)
        pi_smooth[k]  = pi_filt[k] * (G @ ratio)
        pi_smooth[k] /= pi_smooth[k].sum()  # normalise

    return pi_smooth


pi_smooth = bayesian_smooth(pi_pred, pi_filt, G)

print('Smoothing results (T = 4):')
print(f"{'k':>3}  {'y_k':>10}  {'pi_k^T(Bull)':>14}  {'pi_k^T(Neut)':>14}  {'pi_k^T(Bear)':>14}")
print('-' * 60)
for k in range(T_fs):
    y = obs_names[obs_seq_fs[k]]
    print(f"{k+1:>3}  {y:>10}  {pi_smooth[k,0]:>14.4f}  {pi_smooth[k,1]:>14.4f}  {pi_smooth[k,2]:>14.4f}")

print()
print(f'Verify pi_T^T == pi_T: {np.allclose(pi_smooth[-1], pi_filt[-1])}')
print()
print('Smoother vs filter (difference pi_k^T - pi_k):')
diff = pi_smooth - pi_filt
for k in range(T_fs):
    print(f"  k={k+1}: {diff[k]}")

In [ ]:
# Smoothing distributions pi_k^T (solid bars) vs filter pi_k (hatched reference bars)
display(Image('smoothing_distributions.png'))

## 4. Viterbi Algorithm

The goal is to compute the **MAP path estimator**:
$$x^*_{0:T} = \arg\max_{x_{0:T}}\; p(x_{0:T} \mid y_{1:T})$$

This is the *jointly* most probable hidden-state sequence — it differs from independently taking $\arg\max_{x_k} \pi_k^T(x_k)$ at each step (the marginal MAP).

### Viterbi Recursion

Define $V_k(x_k) := \max_{x_0,\ldots,x_{k-1}} p(x_0,\ldots,x_k,\, y_1,\ldots,y_k)$.

**Forward pass:**
$$V_1(x_1) = p(y_1 \mid x_1)\, p(x_1), \qquad V_k(x_k) = p(y_k \mid x_k) \cdot \max_{x_{k-1}} \left[ p(x_k \mid x_{k-1}) \cdot V_{k-1}(x_{k-1}) \right]$$

Store the argmax pointer: $\hat{x}_k(x_k) = \arg\max_{x_{k-1}} \left[ p(x_k \mid x_{k-1}) \cdot V_{k-1}(x_{k-1}) \right]$

**Backward pass:**
$$x^*_T = \arg\max_{x_T} V_T(x_T), \qquad x^*_k = \hat{x}_{k+1}(x^*_{k+1}) \quad \text{for } k = T-1, \ldots, 1$$

### Log-Domain Computation

Working in log-domain avoids numerical underflow. Define $\ell_k(x_k) := \log V_k(x_k)$:
$$\ell_k(j) = \log H_{j,\,y_k} + \max_{i} \left[ \log G_{ij} + \ell_{k-1}(i) \right]$$

The argmax pointer $\hat{x}_k(j)$ is unchanged. Final decoding: $x^*_T = \arg\max_j\, \ell_T(j)$, then trace back via $\hat{x}$.

In [ ]:
def viterbi(obs_seq, pi_0, G, H):
    """
    Viterbi algorithm in log-domain.

    Parameters
    ----------
    obs_seq : list[int]   observation indices, length T
    pi_0    : ndarray(n)  initial distribution
    G       : ndarray(n,n) transition matrix, G[i,j] = p(X_k=j | X_{k-1}=i)
    H       : ndarray(n,m) emission matrix,   H[i,o] = p(Y_k=o | X_k=i)

    Returns
    -------
    path  : ndarray(T, int)  optimal state sequence x*_{1:T}
    log_V : ndarray(T, n)    log Viterbi values log V_k(x)
    psi   : ndarray(T, n)    argmax back-pointers hat_x_k(x)
    """
    T, n = len(obs_seq), len(pi_0)
    log_V = np.full((T, n), -np.inf)
    psi   = np.zeros((T, n), dtype=int)

    # Initialise: V_1(j) = pi_0(j) * H[j, y_1]
    log_V[0] = np.log(pi_0) + np.log(H[:, obs_seq[0]])

    # Forward pass
    for k in range(1, T):
        # scores[i, j] = log V_{k-1}(i) + log G[i, j]
        scores   = log_V[k-1, :, None] + np.log(G)   # shape (n, n)
        psi[k]   = np.argmax(scores, axis=0)           # best predecessor for each j
        log_V[k] = np.max(scores, axis=0) + np.log(H[:, obs_seq[k]])

    # Backward pass
    path     = np.zeros(T, dtype=int)
    path[-1] = np.argmax(log_V[-1])
    for k in range(T - 2, -1, -1):
        path[k] = psi[k+1, path[k+1]]

    return path, log_V, psi


obs_seq = obs_seq_full
T_v = len(obs_seq)
path, log_V, psi = viterbi(obs_seq, pi_0, G, H)
V = np.exp(log_V)

print('Viterbi results (T = 8):')
print(f"{'k':>3}  {'y_k':>10}  {'x*_k':>8}  {'V_k(x*_k)':>14}  {'back-ptr':>10}")
print('-' * 55)
for k in range(T_v):
    y = obs_names[obs_seq[k]]
    x = states[path[k]]
    v = V[k, path[k]]
    bp = states[psi[k, path[k]]] if k > 0 else '—'
    print(f"{k+1:>3}  {y:>10}  {x:>8}  {v:>14.4e}  {bp:>10}")

print()
print('Optimal Viterbi path:')
print('  ' + '  →  '.join(states[s] for s in path))

In [ ]:
# Viterbi trellis (T = 8): amber = Viterbi path, cyan = marginal MAP path
# Nodes show V_k(x) values; dashed lines show stored argmax pointers
display(Image('viterbi_regimes.png'))

### Marginal MAP vs. Joint MAP

The **marginal MAP** at each step independently maximises the smoothing marginal:
$$\tilde{x}_k = \arg\max_{x_k}\; \pi_k^T(x_k) = \arg\max_{x_k}\; p(x_k \mid y_{1:T})$$

This sequence $(\tilde{x}_k)_{k=1}^T$ need **not** equal the joint MAP path $(x^*_k)_{k=1}^T$ from Viterbi, because:

- The **Viterbi path** $x^*_{0:T}$ maximises the *joint* posterior $p(x_{0:T} \mid y_{1:T})$, so every consecutive pair $(x^*_k, x^*_{k+1})$ is jointly coherent.
- The **marginal MAP** independently maximises each $p(x_k \mid y_{1:T})$, which ignores how likely the transitions between states are.

The cyan dashed path in the trellis figure shows the marginal MAP. It differs from the Viterbi path at time steps where a locally most-probable state is reached via an unlikely transition.

In [ ]:
# Compute marginal MAP for comparison (forward-backward)
from scipy.special import logsumexp

log_H = np.log(H)
log_G = np.log(G)

# Forward (alpha) pass
log_alpha = np.full((T_v, n_states), -np.inf)
log_alpha[0] = np.log(pi_0) + log_H[:, obs_seq[0]]
for k in range(1, T_v):
    log_alpha[k] = log_H[:, obs_seq[k]] + logsumexp(
        log_alpha[k-1, :, None] + log_G, axis=0)

# Backward (beta) pass
log_beta = np.zeros((T_v, n_states))
for k in range(T_v - 2, -1, -1):
    log_beta[k] = logsumexp(
        log_G + log_H[:, obs_seq[k+1]] + log_beta[k+1], axis=1)

# Smoothed marginals gamma_k(i) = p(X_k = i | y_{1:T})
log_gamma = log_alpha + log_beta
log_gamma -= logsumexp(log_gamma, axis=1, keepdims=True)
marginal_path = np.argmax(log_gamma, axis=1)

print('Comparison: Viterbi (joint MAP) vs. Marginal MAP')
print(f"{'k':>3}  {'y_k':>10}  {'Viterbi x*_k':>14}  {'Marginal argmax':>16}  {'same?':>6}")
print('-' * 57)
for k in range(T_v):
    y  = obs_names[obs_seq[k]]
    xv = states[path[k]]
    xm = states[marginal_path[k]]
    ok = '✓' if path[k] == marginal_path[k] else '✗'
    print(f"{k+1:>3}  {y:>10}  {xv:>14}  {xm:>16}  {ok:>6}")

### Viterbi Trellis Animation

The animation below shows the Viterbi algorithm on the first $T = 6$ observations:

- **Forward pass** (frames 1–6): the trellis is built column by column. For each new time step $k$, all incoming edges (gray) are evaluated, the winning predecessor per state is highlighted (coloured edge), the $V_k(x)$ value appears in the node, and the back-pointer $\hat{x}_k(x)$ is stored below.
- **Pause**: the complete trellis with all stored back-pointers.
- **Backward pass** (frames 7–12): starting from $x^*_T = \arg\max_x V_T(x)$, the algorithm traces back via the stored pointers, revealing the optimal path from right to left. Each newly selected node gets an amber ring and a callout label.

In [ ]:
# Animated GIF — plays automatically in Jupyter
display(Image('viterbi_trellis.gif'))

## 5. Comparison

All three algorithms operate on the same HMM and the same observation record $y_{1:T}$, but answer different inference questions:

| | **Filter** $\pi_k$ | **Smoother** $\pi_k^T$ | **Viterbi** |
|---|---|---|---|
| **Uses data** | $y_{1:k}$ (past + present) | $y_{1:T}$ (all) | $y_{1:T}$ (all) |
| **Online** | ✓ | ✗ | ✗ |
| **Output** | distribution over $x_k$ | distribution over $x_k$ | point estimate of $x_{0:T}$ |
| **Formula** | $p(x_k \mid y_{1:k})$ | $p(x_k \mid y_{1:T})$ | $\arg\max_{x_{0:T}} p(x_{0:T} \mid y_{1:T})$ |
| **Complexity** | $O(T n^2)$ | $O(T n^2)$ | $O(T n^2)$ |
| **Pass** | one forward pass | forward + backward | forward + backward |

**Key distinctions:**

- The **smoother** uses future observations to refine the estimate of past states — it is always at least as accurate as the filter in terms of posterior uncertainty.
- The **Viterbi** path is the *jointly* most probable sequence; marginal argmax $\arg\max_{x_k} \pi_k^T(x_k)$ can differ because it ignores transition probabilities between time steps.
- All three recursions have the same $O(Tn^2)$ cost for finite-state models with $n$ states and $T$ time steps.